<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/CNN_CODE2_%EB%B9%A8%EA%B0%95%EC%83%89%EC%9E%90%EB%8F%99%EC%B0%A8_%EC%9D%B8%EC%8B%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1번 코드 📦 Google Colab용 실제 사진 자동차 인식 CNN 완전 코드


# 필요한 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os

# 코랩에서 한글 폰트 설정 (경고 무시)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 라이브러리 로드 완료!")

# =============================================================================
# 자동차 인식 CNN 클래스
# =============================================================================

class ColabCarCNN:
    """코랩용 자동차 인식 CNN (실제 사진 업로드 방식)"""

    def __init__(self):
        print("🚗 실제 사진 업로드용 자동차 인식 CNN 시작!")

    def load_real_car_photo(self, image_path, size=224):
        """실제 자동차 사진 로드"""
        print(f"📷 실제 자동차 사진 로드 중: {image_path}")

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"이미지를 찾을 수 없습니다: {image_path}")

        # BGR을 RGB로 변환 (OpenCV는 BGR 순서)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 크기 조정
        original_height, original_width = image.shape[:2]
        image = cv2.resize(image, (size, size))

        print(f"✅ 이미지 로드 완료: {original_width}x{original_height} → {size}x{size}")
        return image

    def rgb_to_grayscale(self, rgb_image):
        """RGB를 그레이스케일로 변환"""
        # 표준 공식 사용
        gray = np.dot(rgb_image[...,:3], [0.299, 0.587, 0.114])
        return gray.astype(np.uint8)

    def convolution_2d(self, image, kernel, stride=1):
        """2D 합성곱 연산"""
        if len(image.shape) == 3:
            image = self.rgb_to_grayscale(image)

        input_h, input_w = image.shape
        kernel_h, kernel_w = kernel.shape

        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1

        output = np.zeros((output_h, output_w))

        for i in range(0, output_h * stride, stride):
            for j in range(0, output_w * stride, stride):
                if i + kernel_h <= input_h and j + kernel_w <= input_w:
                    window = image[i:i+kernel_h, j:j+kernel_w]
                    output[i//stride, j//stride] = np.sum(window * kernel)

        return output

    def max_pooling(self, feature_map, pool_size=2):
        """맥스 풀링"""
        input_h, input_w = feature_map.shape
        output_h = input_h // pool_size
        output_w = input_w // pool_size

        output = np.zeros((output_h, output_w))

        for i in range(output_h):
            for j in range(output_w):
                start_i = i * pool_size
                start_j = j * pool_size
                pool_region = feature_map[start_i:start_i+pool_size, start_j:start_j+pool_size]
                output[i, j] = np.max(pool_region)

        return output

    def analyze_car_features(self, image, results):
        """실제 이미지에서 자동차 특징 분석"""
        print("\n🔍 실제 이미지 자동차 특징 분석 중...")

        # 1. 색상 분석 (RGB 채널별)
        red_channel = np.mean(image[:,:,0])
        green_channel = np.mean(image[:,:,1])
        blue_channel = np.mean(image[:,:,2])

        print(f"   RGB 평균값: R={red_channel:.1f}, G={green_channel:.1f}, B={blue_channel:.1f}")

        # 2. 엣지 강도 분석
        edge_intensity = np.mean(np.abs(results['conv1']))
        edge_std = np.std(results['conv1'])

        print(f"   엣지 강도: 평균={edge_intensity:.2f}, 표준편차={edge_std:.2f}")

        # 3. 형태 분석
        final_features = results['final'].flatten()
        shape_complexity = np.std(final_features)
        feature_concentration = np.mean(np.abs(final_features))

        print(f"   형태 복잡도: {shape_complexity:.2f}")
        print(f"   특징 집중도: {feature_concentration:.2f}")

        # 4. 실제 이미지 기반 자동차 점수 계산
        car_score = 0.1  # 기본 점수

        # 자동차 색상 특징 (금속성, 페인트 느낌)
        color_variance = np.var([red_channel, green_channel, blue_channel])
        if color_variance > 200:  # 색상 대비가 뚜렷함
            car_score += 0.2

        # 엣지가 뚜렷하면서 적당히 복잡함 (자동차 윤곽)
        if 20 < edge_intensity < 80 and edge_std > 10:
            car_score += 0.3

        # 붉은색 계열이 강하면 (빨간 자동차)
        if red_channel > green_channel and red_channel > blue_channel:
            car_score += 0.25

        # 형태가 적당히 복잡함 (자동차는 여러 부품)
        if 5 < shape_complexity < 100:
            car_score += 0.2

        # 특징이 중앙에 집중됨 (자동차 몸체)
        center_h, center_w = results['final'].shape
        center_region = results['final'][center_h//3:2*center_h//3, center_w//3:2*center_w//3]
        center_strength = np.mean(np.abs(center_region))
        if center_strength > feature_concentration * 0.8:
            car_score += 0.15

        # 가로가 세로보다 긴 형태 (일반적인 자동차)
        aspect_ratio = results['final'].shape[1] / results['final'].shape[0]
        if 1.2 < aspect_ratio < 2.5:
            car_score += 0.1

        print(f"   자동차 특징 점수: {car_score:.3f}")

        return min(max(car_score, 0.05), 0.95)

    def run_cnn_pipeline(self, image):
        """CNN 파이프라인 실행"""
        print("\n🔍 실제 사진 CNN 파이프라인 시작!")

        results = {}
        current_image = image.copy()

        # 1. 원본 저장
        results['original'] = current_image
        print(f"1. 원본 이미지: {current_image.shape}")

        # 2. 첫 번째 합성곱 (수직 엣지 검출)
        print("\n2. 수직 엣지 검출 필터 적용...")
        vertical_filter = np.array([
            [-1, 0, 1],
            [-2, 0, 2],
            [-1, 0, 1]
        ])
        conv1 = self.convolution_2d(current_image, vertical_filter)
        results['conv1'] = conv1
        print(f"   결과: {conv1.shape}")

        # 3. 첫 번째 풀링
        print("\n3. 첫 번째 맥스 풀링...")
        pool1 = self.max_pooling(conv1)
        results['pool1'] = pool1
        print(f"   결과: {pool1.shape}")

        # 4. 두 번째 합성곱 (수평 엣지 검출)
        print("\n4. 수평 엣지 검출 필터 적용...")
        horizontal_filter = np.array([
            [-1, -2, -1],
            [ 0,  0,  0],
            [ 1,  2,  1]
        ])
        conv2 = self.convolution_2d(pool1, horizontal_filter)
        results['conv2'] = conv2
        print(f"   결과: {conv2.shape}")

        # 5. 두 번째 풀링
        print("\n5. 두 번째 맥스 풀링...")
        pool2 = self.max_pooling(conv2)
        results['pool2'] = pool2
        print(f"   결과: {pool2.shape}")

        # 6. 몇 단계 더 (7x7까지 줄이기)
        current = pool2
        layer_count = 3
        while current.shape[0] > 7 and layer_count < 6:
            print(f"\n{layer_count + 3}. 추가 합성곱층...")
            simple_filter = np.array([
                [1, 1, 1],
                [1, -8, 1],
                [1, 1, 1]
            ])
            current = self.convolution_2d(current, simple_filter)
            print(f"   합성곱 결과: {current.shape}")

            if current.shape[0] > 14:
                current = self.max_pooling(current)
                print(f"   풀링 결과: {current.shape}")

            layer_count += 1

        results['final'] = current

        # 7. 실제 이미지 기반 분류
        print(f"\n7. 실제 이미지 기반 분류...")
        flattened = current.flatten()
        print(f"   평탄화: {current.shape} -> {flattened.shape}")

        # 실제 이미지 특징 분석
        car_prob = self.analyze_car_features(image, results)

        # 나머지 클래스 확률 분배
        remaining = 1 - car_prob
        pedestrian_prob = remaining * 0.2   # 20%
        traffic_light_prob = remaining * 0.3 # 30%
        sign_prob = remaining * 0.5          # 50%

        results['predictions'] = [car_prob, pedestrian_prob, traffic_light_prob, sign_prob]
        results['class_names'] = ['Car', 'Pedestrian', 'Traffic Light', 'Traffic Sign']

        print(f"\n🎯 실제 이미지 분석 결과:")
        for name, prob in zip(results['class_names'], results['predictions']):
            print(f"   {name}: {prob:.1%}")

        return results

    def visualize_results(self, results):
        """결과 시각화"""
        print("\n📊 결과 시각화 중...")

        # 큰 figure 생성
        fig = plt.figure(figsize=(16, 12))

        # 2x3 서브플롯 설정

        # 1. 원본 이미지
        plt.subplot(2, 3, 1)
        plt.imshow(results['original'])
        plt.title('1. Original Image\n(Uploaded Photo)', fontsize=12, fontweight='bold')
        plt.axis('off')

        # 2. 첫 번째 합성곱 (수직 엣지)
        plt.subplot(2, 3, 2)
        plt.imshow(results['conv1'], cmap='gray')
        plt.title(f'2. Vertical Edge Detection\n{results["conv1"].shape}', fontsize=12)
        plt.axis('off')

        # 3. 첫 번째 풀링
        plt.subplot(2, 3, 3)
        plt.imshow(results['pool1'], cmap='gray')
        plt.title(f'3. First Max Pooling\n{results["pool1"].shape}', fontsize=12)
        plt.axis('off')

        # 4. 두 번째 합성곱 (수평 엣지)
        plt.subplot(2, 3, 4)
        plt.imshow(results['conv2'], cmap='gray')
        plt.title(f'4. Horizontal Edge Detection\n{results["conv2"].shape}', fontsize=12)
        plt.axis('off')

        # 5. 두 번째 풀링
        plt.subplot(2, 3, 5)
        plt.imshow(results['pool2'], cmap='gray')
        plt.title(f'5. Second Max Pooling\n{results["pool2"].shape}', fontsize=12)
        plt.axis('off')

        # 6. 최종 특징맵
        plt.subplot(2, 3, 6)
        plt.imshow(results['final'], cmap='viridis')
        plt.title(f'6. Final Feature Map\n{results["final"].shape}', fontsize=12)
        plt.axis('off')

        plt.suptitle('🚗 Real Photo Car Recognition CNN Pipeline', fontsize=16, fontweight='bold', y=0.98)
        plt.tight_layout()
        plt.show()

        # 예측 결과 막대 그래프
        plt.figure(figsize=(12, 6))
        colors = ['red', 'blue', 'orange', 'green']
        bars = plt.bar(results['class_names'], results['predictions'], color=colors, alpha=0.8)

        # 가장 높은 확률의 막대를 더 진하게
        max_idx = np.argmax(results['predictions'])
        bars[max_idx].set_alpha(1.0)
        bars[max_idx].set_edgecolor('black')
        bars[max_idx].set_linewidth(3)

        plt.title('🎯 Real Photo Analysis Results', fontsize=16, fontweight='bold')
        plt.ylabel('Probability', fontsize=12)
        plt.ylim(0, 1)

        # 막대 위에 퍼센트 표시 (더 크고 명확하게)
        for i, (bar, prob) in enumerate(zip(bars, results['predictions'])):
            color = 'white' if i == max_idx else 'black'
            fontweight = 'bold' if i == max_idx else 'normal'
            fontsize = 14 if i == max_idx else 11

            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.1%}', ha='center', va='bottom',
                    fontweight=fontweight, fontsize=fontsize, color=color)

        # 최고 확률 클래스 표시
        best_class = results['class_names'][max_idx]
        best_prob = results['predictions'][max_idx]
        plt.text(0.02, 0.98, f'🏆 Prediction: {best_class} ({best_prob:.1%})',
                transform=plt.gca().transAxes, fontsize=14, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    def print_summary(self, results):
        """요약 출력"""
        print("\n" + "="*60)
        print("📋 Real Photo CNN Pipeline Summary")
        print("="*60)

        steps = [
            ("Original Image", results['original'].shape, "Uploaded photo"),
            ("Vertical Edge Detection", results['conv1'].shape, "Car outline extraction"),
            ("First Pooling", results['pool1'].shape, "Size reduced by 50%"),
            ("Horizontal Edge Detection", results['conv2'].shape, "Horizontal line features"),
            ("Second Pooling", results['pool2'].shape, "Size reduced by 50% again"),
            ("Final Feature Map", results['final'].shape, "Highly compressed features")
        ]
        for i, (step_name, shape, description) in enumerate(steps, 1):
            if len(shape) == 3:
                shape_str = f"{shape[0]}×{shape[1]}×{shape[2]}"
            else:
                shape_str = f"{shape[0]}×{shape[1]}"
            print(f"{i}. {step_name:25}: {shape_str:12} - {description}")

        # 최종 결과
        best_class = results['class_names'][np.argmax(results['predictions'])]
        best_prob = max(results['predictions'])

        print(f"\n🎯 Final Result: {best_class} ({best_prob:.1%} confidence)")
        print("="*60)

# =============================================================================
# 실행 함수들
# =============================================================================

def upload_and_test():
    """파일 업로드 후 테스트 - 메인 함수"""
    print("🚀 실제 사진 업로드 자동차 인식 시작!")
    print("="*60)

    # 파일 업로드
    print("📁 자동차 사진을 업로드해주세요...")
    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        print("❌ 파일이 업로드되지 않았습니다!")
        return None

    # 업로드된 첫 번째 파일 사용
    filename = list(uploaded.keys())[0]
    print(f"📷 업로드된 파일: {filename}")

    try:
        # CNN 객체 생성
        cnn = ColabCarCNN()

        # 실제 이미지 로드
        car_image = cnn.load_real_car_photo(filename, size=128)

        # CNN 파이프라인 실행
        results = cnn.run_cnn_pipeline(car_image)

        # 결과 시각화
        cnn.visualize_results(results)

        # 요약 출력
        cnn.print_summary(results)

        print("\n✅ 실제 사진 분석 완료!")
        return results

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        print("💡 파일 형식을 확인해주세요. (jpg, png, jpeg 등)")
        return None

def multiple_photos_test():
    """여러 사진 연속 테스트"""
    print("📁 여러 장의 자동차 사진을 업로드해주세요...")
    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        print("❌ 파일이 업로드되지 않았습니다!")
        return

    cnn = ColabCarCNN()
    results_list = []

    for filename in uploaded.keys():
        print(f"\n{'='*60}")
        print(f"📷 분석 중: {filename}")
        print(f"{'='*60}")

        try:
            # 이미지 로드 및 분석
            car_image = cnn.load_real_car_photo(filename, size=128)
            results = cnn.run_cnn_pipeline(car_image)

            # 간단한 시각화 (공간 절약)
            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(results['original'])
            plt.title(f'Original: {filename}')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(results['final'], cmap='viridis')
            plt.title('Final Features')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            bars = plt.bar(results['class_names'], results['predictions'],
                          color=['red', 'blue', 'orange', 'green'], alpha=0.8)
            max_idx = np.argmax(results['predictions'])
            bars[max_idx].set_alpha(1.0)
            best_class = results['class_names'][max_idx]
            best_prob = results['predictions'][max_idx]
            plt.title(f'Result: {best_class} ({best_prob:.1%})')
            plt.ylabel('Probability')

            plt.tight_layout()
            plt.show()

            results_list.append((filename, best_class, best_prob))

        except Exception as e:
            print(f"❌ {filename} 처리 실패: {e}")

    # 전체 결과 요약
    print(f"\n{'='*60}")
    print("📋 전체 분석 결과 요약")
    print(f"{'='*60}")
    for filename, pred_class, confidence in results_list:
        print(f"📷 {filename:20} → {pred_class:15} ({confidence:.1%})")

# =============================================================================
# 메인 실행 부분
# =============================================================================

print("🎯 실행 옵션:")
print("1. 단일 사진 분석: upload_and_test()")
print("2. 여러 사진 분석: multiple_photos_test()")
print("\n💡 추천: upload_and_test() 로 시작하세요!")

# 자동 실행 (실제 사진 업로드 방식)
print("\n🚀 실제 사진 업로드 자동차 인식 시작...")
print("📁 아래에서 자동차 사진을 선택해주세요!")
results = upload_and_test()

1번코드에 input image 추가

In [ ]:
# 📦 Google Colab용 실제 사진 자동차 인식 CNN 완전 코드
# 이 코드를 코랩 셀에 복사해서 한 번에 실행하세요!

# 필요한 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os

# 코랩에서 한글 폰트 설정 (경고 무시)
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 라이브러리 로드 완료!")

# =============================================================================
# 자동차 인식 CNN 클래스
# =============================================================================

class ColabCarCNN:
    """코랩용 자동차 인식 CNN (실제 사진 업로드 방식)"""

    def __init__(self):
        print("🚗 실제 사진 업로드용 자동차 인식 CNN 시작!")

    def load_real_car_photo(self, image_path, size=224):
        """실제 자동차 사진 로드 (원본과 입력 버전 모두 저장)"""
        print(f"📷 실제 자동차 사진 로드 중: {image_path}")

        # 이미지 로드
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"이미지를 찾을 수 없습니다: {image_path}")

        # BGR을 RGB로 변환 (OpenCV는 BGR 순서)
        original_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 원본 크기 저장
        original_height, original_width = original_rgb.shape[:2]

        # CNN 입력용 크기 조정
        resized_image = cv2.resize(original_rgb, (size, size))

        print(f"✅ 이미지 로드 완료:")
        print(f"   원본 크기: {original_width}x{original_height}")
        print(f"   CNN 입력 크기: {size}x{size}")

        return original_rgb, resized_image

    def rgb_to_grayscale(self, rgb_image):
        """RGB를 그레이스케일로 변환"""
        # 표준 공식 사용
        gray = np.dot(rgb_image[...,:3], [0.299, 0.587, 0.114])
        return gray.astype(np.uint8)

    def convolution_2d(self, image, kernel, stride=1):
        """2D 합성곱 연산"""
        if len(image.shape) == 3:
            image = self.rgb_to_grayscale(image)

        input_h, input_w = image.shape
        kernel_h, kernel_w = kernel.shape

        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1

        output = np.zeros((output_h, output_w))

        for i in range(0, output_h * stride, stride):
            for j in range(0, output_w * stride, stride):
                if i + kernel_h <= input_h and j + kernel_w <= input_w:
                    window = image[i:i+kernel_h, j:j+kernel_w]
                    output[i//stride, j//stride] = np.sum(window * kernel)

        return output

    def max_pooling(self, feature_map, pool_size=2):
        """맥스 풀링"""
        input_h, input_w = feature_map.shape
        output_h = input_h // pool_size
        output_w = input_w // pool_size

        output = np.zeros((output_h, output_w))

        for i in range(output_h):
            for j in range(output_w):
                start_i = i * pool_size
                start_j = j * pool_size
                pool_region = feature_map[start_i:start_i+pool_size, start_j:start_j+pool_size]
                output[i, j] = np.max(pool_region)

        return output

    def analyze_car_features(self, image, results):
        """실제 이미지에서 자동차 특징 분석"""
        print("\n🔍 실제 이미지 자동차 특징 분석 중...")

        # 1. 색상 분석 (RGB 채널별)
        red_channel = np.mean(image[:,:,0])
        green_channel = np.mean(image[:,:,1])
        blue_channel = np.mean(image[:,:,2])

        print(f"   RGB 평균값: R={red_channel:.1f}, G={green_channel:.1f}, B={blue_channel:.1f}")

        # 2. 엣지 강도 분석
        edge_intensity = np.mean(np.abs(results['conv1']))
        edge_std = np.std(results['conv1'])

        print(f"   엣지 강도: 평균={edge_intensity:.2f}, 표준편차={edge_std:.2f}")

        # 3. 형태 분석
        final_features = results['final'].flatten()
        shape_complexity = np.std(final_features)
        feature_concentration = np.mean(np.abs(final_features))

        print(f"   형태 복잡도: {shape_complexity:.2f}")
        print(f"   특징 집중도: {feature_concentration:.2f}")

        # 4. 실제 이미지 기반 자동차 점수 계산
        car_score = 0.1  # 기본 점수

        # 자동차 색상 특징 (금속성, 페인트 느낌)
        color_variance = np.var([red_channel, green_channel, blue_channel])
        if color_variance > 200:  # 색상 대비가 뚜렷함
            car_score += 0.2

        # 엣지가 뚜렷하면서 적당히 복잡함 (자동차 윤곽)
        if 20 < edge_intensity < 80 and edge_std > 10:
            car_score += 0.3

        # 붉은색 계열이 강하면 (빨간 자동차)
        if red_channel > green_channel and red_channel > blue_channel:
            car_score += 0.25

        # 형태가 적당히 복잡함 (자동차는 여러 부품)
        if 5 < shape_complexity < 100:
            car_score += 0.2

        # 특징이 중앙에 집중됨 (자동차 몸체)
        center_h, center_w = results['final'].shape
        center_region = results['final'][center_h//3:2*center_h//3, center_w//3:2*center_w//3]
        center_strength = np.mean(np.abs(center_region))
        if center_strength > feature_concentration * 0.8:
            car_score += 0.15

        # 가로가 세로보다 긴 형태 (일반적인 자동차)
        aspect_ratio = results['final'].shape[1] / results['final'].shape[0]
        if 1.2 < aspect_ratio < 2.5:
            car_score += 0.1

        print(f"   자동차 특징 점수: {car_score:.3f}")

        return min(max(car_score, 0.05), 0.95)

    def run_cnn_pipeline(self, original_image, cnn_input_image):
        """CNN 파이프라인 실행 (원본과 입력 이미지 분리)"""
        print("\n🔍 실제 사진 CNN 파이프라인 시작!")

        results = {}

        # 원본과 CNN 입력 이미지 모두 저장
        results['original_photo'] = original_image  # 업로드된 원본
        results['original'] = cnn_input_image       # CNN 처리용 (크기 조정됨)

        print(f"1. 원본 사진: {original_image.shape}")
        print(f"2. CNN 입력 이미지: {cnn_input_image.shape}")

        # 3. 첫 번째 합성곱 (수직 엣지 검출)
        print("\n3.   conv1  수직 엣지 검출 필터 적용...")
        vertical_filter = np.array([
            [-1, 0, 1],
            [-2, 0, 2],
            [-1, 0, 1]
        ])
        conv1 = self.convolution_2d(cnn_input_image, vertical_filter)
        results['conv1'] = conv1
        print(f"   결과: {conv1.shape}")

        # 4. 첫 번째 풀링
        print("\n4. 첫 번째 맥스 풀링...")
        pool1 = self.max_pooling(conv1)
        results['pool1'] = pool1
        print(f"   결과: {pool1.shape}")

        # 5. 두 번째 합성곱 (수평 엣지 검출)
        print("\n5. 수평 엣지 검출 필터 적용...")
        horizontal_filter = np.array([
            [-1, -2, -1],
            [ 0,  0,  0],
            [ 1,  2,  1]
        ])
        conv2 = self.convolution_2d(pool1, horizontal_filter)
        results['conv2'] = conv2
        print(f"   결과: {conv2.shape}")

        # 6. 두 번째 풀링
        print("\n6. 두 번째 맥스 풀링...")
        pool2 = self.max_pooling(conv2)
        results['pool2'] = pool2
        print(f"   결과: {pool2.shape}")

        # 7. 몇 단계 더 (7x7까지 줄이기)
        current = pool2
        layer_count = 3
        while current.shape[0] > 7 and layer_count < 6:
            print(f"\n{layer_count + 4}. 추가 합성곱층...")
            simple_filter = np.array([
                [1, 1, 1],
                [1, -8, 1],
                [1, 1, 1]
            ])
            current = self.convolution_2d(current, simple_filter)
            print(f"   합성곱 결과: {current.shape}")

            if current.shape[0] > 14:
                current = self.max_pooling(current)
                print(f"   풀링 결과: {current.shape}")

            layer_count += 1

        results['final'] = current

        # 8. 실제 이미지 기반 분류
        print(f"\n8. 실제 이미지 기반 분류...")
        flattened = current.flatten()
        print(f"   평탄화: {current.shape} -> {flattened.shape}")

        # 실제 이미지 특징 분석 (CNN 입력 이미지 사용)
        car_prob = self.analyze_car_features(cnn_input_image, results)

        # 나머지 클래스 확률 분배
        remaining = 1 - car_prob
        pedestrian_prob = remaining * 0.2   # 20%
        traffic_light_prob = remaining * 0.3 # 30%
        sign_prob = remaining * 0.5          # 50%

        results['predictions'] = [car_prob, pedestrian_prob, traffic_light_prob, sign_prob]
        results['class_names'] = ['Car', 'Pedestrian', 'Traffic Light', 'Traffic Sign']

        print(f"\n🎯 실제 이미지 분석 결과:")
        for name, prob in zip(results['class_names'], results['predictions']):
            print(f"   {name}: {prob:.1%}")

        return results

    def visualize_results(self, results):
        """결과 시각화 (원본과 입력 이미지 구분)"""
        print("\n📊 결과 시각화 중...")

        # 더 큰 figure 생성 (3x3 그리드)
        fig = plt.figure(figsize=(18, 16))

     # 1-3번은 전처리 과정
        print("\n🧠✨ 1-3번은 전처리 과정 ")
        print("\n🧠✨원본 이미지 (업로드된 실제 사진) - 큰 크기 ")
        # 1. 원본 이미지 (업로드된 실제 사진) - 큰 크기
        plt.subplot(3, 3, 1)
        plt.imshow(results['original_photo'])  # 실제 원본 사용
        original_size = results['original_photo'].shape
        plt.title(f'1. Original Photo\n{original_size[1]}×{original_size[0]} (Uploaded)', fontsize=11, fontweight='bold', pad=15)
        plt.axis('off')

        # 2. CNN 입력 이미지 (크기 조정된 버전) - 작은 크기 + 빨간 테두리
        print("\n🧠✨2. CNN 입력 이미지 (크기 조정된 버전) - 작은 크기 + 빨간 테두리 ")
        plt.subplot(3, 3, 2)
        cnn_input = results['original']  # 리사이즈된 이미지
        plt.imshow(cnn_input)

        # 빨간 경계선으로 리사이즈 강조
        ax = plt.gca()
        ax.add_patch(plt.Rectangle((0, 0), cnn_input.shape[1]-1, cnn_input.shape[0]-1,
                                 fill=False, edgecolor='red', linewidth=3))
        cnn_size = cnn_input.shape
        plt.title(f'2. CNN Input Image\n{cnn_size[1]}×{cnn_size[0]} (Resized)', fontsize=11, fontweight='bold', color='red', pad=10)
        plt.axis('off')

        # 3. 그레이스케일 변환
        print("\n🧠✨3.그레이스케일 변환")
        plt.subplot(3, 3, 3)
        gray_image = self.rgb_to_grayscale(results['original'])
        plt.imshow(gray_image, cmap='gray')
        plt.title(f'3. Grayscale\n{gray_image.shape[1]}×{gray_image.shape[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 4. 첫 번째 합성곱 (수직 엣지)  입력 크기: 128×128
        # 필터 크기: 3×3
        #패딩: 없음 (Valid padding)
        #출력 크기 = (입력 - 필터 + 1) = (128 - 3 + 1) = 126×126
        print("\n🧠✨3.첫 번째 합성곱 (수직 엣지)  입력 크기: 128×128, 필터 크기: 3×3,패딩: 없음 (Valid padding), 출력 크기 = (입력 - 필터 + 1) = (128 - 3 + 1) = 126×126")
        plt.subplot(3, 3, 4)
        plt.imshow(results['conv1'], cmap='gray')
        conv1_size = results['conv1'].shape
        plt.title(f'4.1st Conv (Vertical Edge)\n{conv1_size[1]}×{conv1_size[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 5. 첫 번째 풀링
        plt.subplot(3, 3, 5)
        plt.imshow(results['pool1'], cmap='gray')
        pool1_size = results['pool1'].shape
        plt.title(f'5. First Pooling\n{pool1_size[1]}×{pool1_size[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 6. 두 번째 합성곱 (수평 엣지)
        plt.subplot(3, 3, 6)
        plt.imshow(results['conv2'], cmap='gray')
        conv2_size = results['conv2'].shape
        plt.title(f'6. Horizontal Edge\n{conv2_size[1]}×{conv2_size[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 7. 두 번째 풀링
        plt.subplot(3, 3, 7)
        plt.imshow(results['pool2'], cmap='gray')
        pool2_size = results['pool2'].shape
        plt.title(f'7. Second Pooling\n{pool2_size[1]}×{pool2_size[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 8. 최종 특징맵
        plt.subplot(3, 3, 8)
        plt.imshow(results['final'], cmap='viridis')
        final_size = results['final'].shape
        plt.title(f'8. Final Features\n{final_size[1]}×{final_size[0]}', fontsize=11, pad=15)
        plt.axis('off')

        # 9. 특징맵 히트맵 (추가 정보)
        plt.subplot(3, 3, 9)
        plt.imshow(results['final'], cmap='hot')
        plt.title(f'9. Feature Heatmap\n(Intensity View)', fontsize=11, pad=15)
        plt.axis('off')

        # 전체 제목을 더 위로 올리고 크기 줄임
        plt.suptitle('🚗 CNN Pipeline: Original → Resized → Features', fontsize=14, fontweight='bold', y=0.97)

        # 서브플롯 간격 조정
        plt.subplots_adjust(top=0.93, hspace=0.35, wspace=0.25)
        plt.show()

        # 예측 결과 막대 그래프
        plt.figure(figsize=(12, 6))
        colors = ['red', 'blue', 'orange', 'green']
        bars = plt.bar(results['class_names'], results['predictions'], color=colors, alpha=0.8)

        # 가장 높은 확률의 막대를 더 진하게
        max_idx = np.argmax(results['predictions'])
        bars[max_idx].set_alpha(1.0)
        bars[max_idx].set_edgecolor('black')
        bars[max_idx].set_linewidth(3)

        plt.title('🎯 Real Photo Analysis Results', fontsize=16, fontweight='bold')
        plt.ylabel('Probability', fontsize=12)
        plt.ylim(0, 1)

        # 막대 위에 퍼센트 표시 (더 크고 명확하게)
        for i, (bar, prob) in enumerate(zip(bars, results['predictions'])):
            color = 'white' if i == max_idx else 'black'
            fontweight = 'bold' if i == max_idx else 'normal'
            fontsize = 14 if i == max_idx else 11

            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{prob:.1%}', ha='center', va='bottom',
                    fontweight=fontweight, fontsize=fontsize, color=color)

        # 최고 확률 클래스 표시
        best_class = results['class_names'][max_idx]
        best_prob = results['predictions'][max_idx]
        plt.text(0.02, 0.98, f'🏆 Prediction: {best_class} ({best_prob:.1%})',
                transform=plt.gca().transAxes, fontsize=14, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

    def print_summary(self, results):
        """요약 출력"""
        print("\n" + "="*60)
        print("📋 Real Photo CNN Pipeline Summary")
        print("="*60)

        steps = [
            ("Original Photo", results['original_photo'].shape, "Uploaded photo (full size)"),
            ("CNN Input Image", results['original'].shape, "Resized for CNN processing"),
            ("Vertical Edge Detection", results['conv1'].shape, "Car outline extraction"),
            ("First Pooling", results['pool1'].shape, "Size reduced by 50%"),
            ("Horizontal Edge Detection", results['conv2'].shape, "Horizontal line features"),
            ("Second Pooling", results['pool2'].shape, "Size reduced by 50% again"),
            ("Final Feature Map", results['final'].shape, "Highly compressed features")
        ]
        for i, (step_name, shape, description) in enumerate(steps, 1):
            if len(shape) == 3:
                shape_str = f"{shape[0]}×{shape[1]}×{shape[2]}"
            else:
                shape_str = f"{shape[0]}×{shape[1]}"
            print(f"{i}. {step_name:25}: {shape_str:12} - {description}")

        # 최종 결과
        best_class = results['class_names'][np.argmax(results['predictions'])]
        best_prob = max(results['predictions'])

        print(f"\n🎯 Final Result: {best_class} ({best_prob:.1%} confidence)")
        print("="*60)

# =============================================================================
# 실행 함수들
# =============================================================================

def upload_and_test():
    """파일 업로드 후 테스트 - 메인 함수"""
    print("🚀 실제 사진 업로드 자동차 인식 시작!")
    print("="*60)

    # 파일 업로드
    print("📁 자동차 사진을 업로드해주세요...")
    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        print("❌ 파일이 업로드되지 않았습니다!")
        return None

    # 업로드된 첫 번째 파일 사용
    filename = list(uploaded.keys())[0]
    print(f"📷 업로드된 파일: {filename}")

    try:
        # CNN 객체 생성
        cnn = ColabCarCNN()

        # 실제 이미지 로드
        original_photo, cnn_input = cnn.load_real_car_photo(filename, size=128)

        # CNN 파이프라인 실행
        results = cnn.run_cnn_pipeline(original_photo, cnn_input)

        # 결과 시각화
        cnn.visualize_results(results)

        # 요약 출력
        cnn.print_summary(results)

        print("\n✅ 실제 사진 분석 완료!")
        return results

    except Exception as e:
        print(f"❌ 오류 발생: {e}")
        print("💡 파일 형식을 확인해주세요. (jpg, png, jpeg 등)")
        return None

def multiple_photos_test():
    """여러 사진 연속 테스트"""
    print("📁 여러 장의 자동차 사진을 업로드해주세요...")
    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        print("❌ 파일이 업로드되지 않았습니다!")
        return

    cnn = ColabCarCNN()
    results_list = []

    for filename in uploaded.keys():
        print(f"\n{'='*60}")
        print(f"📷 분석 중: {filename}")
        print(f"{'='*60}")

        try:
            # 이미지 로드 및 분석
            original_photo, cnn_input = cnn.load_real_car_photo(filename, size=128)
            results = cnn.run_cnn_pipeline(original_photo, cnn_input)

            # 간단한 시각화 (공간 절약)
            plt.figure(figsize=(12, 4))

            plt.subplot(1, 3, 1)
            plt.imshow(results['original_photo'])
            plt.title(f'Original: {filename}')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(results['final'], cmap='viridis')
            plt.title('Final Features')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            bars = plt.bar(results['class_names'], results['predictions'],
                          color=['red', 'blue', 'orange', 'green'], alpha=0.8)
            max_idx = np.argmax(results['predictions'])
            bars[max_idx].set_alpha(1.0)
            best_class = results['class_names'][max_idx]
            best_prob = results['predictions'][max_idx]
            plt.title(f'Result: {best_class} ({best_prob:.1%})')
            plt.ylabel('Probability')

            plt.tight_layout()
            plt.show()

            results_list.append((filename, best_class, best_prob))

        except Exception as e:
            print(f"❌ {filename} 처리 실패: {e}")

    # 전체 결과 요약
    print(f"\n{'='*60}")
    print("📋 전체 분석 결과 요약")
    print(f"{'='*60}")
    for filename, pred_class, confidence in results_list:
        print(f"📷 {filename:20} → {pred_class:15} ({confidence:.1%})")

# =============================================================================
# 메인 실행 부분
# =============================================================================

print("🎯 실행 옵션:")
print("1. 단일 사진 분석: upload_and_test()")
print("2. 여러 사진 분석: multiple_photos_test()")
print("\n💡 추천: upload_and_test() 로 시작하세요!")

# 자동 실행 (실제 사진 업로드 방식)
print("\n🚀 실제 사진 업로드 자동차 인식 시작...")
print("📁 아래에서 자동차 사진을 선택해주세요!")
results = upload_and_test()